# Combined CTC + AR training -- sequential only, 80,000 steps

Third leg of the CTC-vs-AR comparison alongside `train_recognizer_ctc_only.ipynb`
and `train_recognizer_ar_only.ipynb`: this one trains **both heads together**
(the original two-head architecture), with the AR decoder using standard
**teacher forcing** the entire run -- unlike the two ablation notebooks'
respective tradeoffs (CTC-only has no AR decoder at all; AR-only is
free-running, no teacher forcing at all).

**Deliberately as close to `train_recognizer_v2_scratch.ipynb` as possible**
-- same setup/data pipeline, same SentencePiece subword tokenizer (+ its
decode-bug patch, section 1a), same `ModelConfig(max_tokens_per_block=16)`.
The one real difference: `sequential_ar_steps` is set equal to `max_steps`
(both 80,000), so `step < sequential_ar_steps` holds for the ENTIRE run --
the AR decoder never leaves plain teacher-forced sequential mode, and
blockwise mode is never reached at all. `max_steps=80,000` (not
`v2_scratch`'s 200,000) to match the other two comparison notebooks'
budget.

**Not standalone** -- like `train_recognizer_v2_scratch.ipynb`, this one
clones/pulls `recognizer`/`real_data` from GitHub at runtime rather than
embedding its own copy. Same platform notes apply: internet access on for
Kaggle, an accelerator selected before running, an `HF_TOKEN` secret
configured.

Checkpoints push to `Panhapich/Tuna-OCR` under `ctc_ar_sequential/` --
alongside `ctc_only_standalone/` and `ar_only_standalone/`, each on its
own prefix so the three runs never collide (see `hf_push.py`'s
`path_prefix`). Saves `latest.pt`/`best.pt`/`metadata.json` only -- see
the other two notebooks' changelog cells for why (no more one file per
`ckpt_every`).

In [ ]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"
REPO_DIR = "tuna-ocr"

def run_git(args):
    """Runs git and raises with git's ACTUAL stderr on failure. A bare
    CalledProcessError only reports "exit status 128", which is git's catch-all
    and says nothing about which of the many possible causes (existing
    directory, auth, network) actually happened."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}):\n{r.stderr.strip()}")
    return r

# Three cases, in order. The middle one is the important fix: after a kernel
# restart the cwd resets to /content (or /kaggle/working), so "recognizer" is no
# longer visible even though a previous run already cloned the repo -- the old
# code then tried to clone again and git aborted with "destination path already
# exists and is not an empty directory" (exit 128).
def pull_latest():
    """Force the clone we're standing in to exactly match origin/main, LOUDLY.
    A stale/diverged clone is the single most confusing failure mode of this
    notebook: the library code is older (or locally modified) vs. what the
    notebook cell driving it assumes, so you get e.g. a TypeError about an
    unexpected keyword argument for something that plainly exists on GitHub.

    This used to be `git pull --ff-only`, which FAILS SILENTLY (just a printed
    warning, not raised) whenever the local clone has diverged from a clean
    fast-forward -- and it always does on Colab/Kaggle, because "restart
    runtime" only restarts the Python kernel, not the filesystem: /content
    (or /kaggle/working) persists across restarts, so a previous session's
    leftover local state (a stray edit, a half-finished git operation, a
    detached HEAD from checking out a specific commit while debugging) sticks
    around and silently blocks every future pull -- so restarting the runtime
    looks like it did nothing, over and over, even though the kernel really
    did restart. This is an ephemeral, notebook-driven clone with no local
    changes ever worth preserving, so there is no real downside to a hard
    reset -- fetch + reset --hard is unconditional and can't get stuck the
    way a fast-forward-only pull can."""
    try:
        run_git(["fetch", "origin"])
        run_git(["reset", "--hard", "origin/main"])
        run_git(["clean", "-fd"])
        print("reset to latest origin/main")
    except RuntimeError as e:
        print("!" * 78)
        print("WARNING: could not update the clone -- running POSSIBLY STALE code.")
        print(f"  {e}")
        print("  If a later cell fails with 'unexpected keyword argument', this is why.")
        print("  This is likely a network issue (offline runtime) since the reset above")
        print("  doesn't fail on local divergence anymore -- check connectivity.")
        print("!" * 78)

if os.path.isdir("recognizer"):
    # Already inside the repo -- which is what re-running this cell in the same
    # session always looks like, since the first run chdir'd here. This branch
    # used to just print and return, so a second run silently kept whatever code
    # the session started with and never saw upstream commits again.
    print(f"already inside the repo working dir: {os.getcwd()}")
    pull_latest()
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"found an existing clone, reusing it: {os.getcwd()}")
    pull_latest()
else:
    run_git(["clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cloned into {os.getcwd()}")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
# Print the resolved commit: the one unambiguous answer to "is my library code
# actually the version I think it is?", checkable against the GitHub history.
print("repo commit:  " + run_git(["log", "-1", "--pretty=%h %s"]).stdout.strip())


In [ ]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that (e.g. via a plain `-r recognizer/requirements.txt`)
# can silently replace it with a build that doesn't match, which breaks GPU support and
# breaks TPU support even harder (torch_xla is pinned to one exact torch version).
# Install everything else normally, and only pip-install torch if it isn't importable
# at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    # Excluding torch from the requirements file isn't a complete guarantee: any
    # dependency in it is free to pull a *different* torch in as its own dependency.
    # On a TPU runtime that's silently fatal -- torch_xla only loads against the exact
    # torch build it was compiled for, and the failure surfaces much later as an opaque
    # import/libtpu error, so check explicitly here rather than discovering it then.
    # importlib.metadata, not `torch.__version__`: torch is already imported in this
    # kernel, so its module object still reports the OLD version no matter what pip
    # just wrote to disk (and importlib.reload(torch) is not a safe way to find out).
    # The distribution metadata reflects what's actually installed now.
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    if torch_after != torch_before:
        print(f"WARNING: pip changed torch {torch_before} -> {torch_after} as a "
              f"transitive dependency. On a TPU runtime, restart the runtime and "
              f"`pip install torch=={torch_before}` before continuing, or torch_xla "
              f"will fail to load.")
    else:
        print(f"using preinstalled torch {torch_after} "
              f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


In [ ]:
import os

from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order. Colab's own secret store (env_utils.get_hf_token) is
# tried first but is NOT reliable: it raises "Secrets can only be fetched when running
# from the Colab UI" whenever the notebook runs detached from the UI tab, which is
# exactly what happened on a long training run here. So fall back to an HF_TOKEN
# environment variable, then to a plain file, then to an interactive prompt --
# deliberately never hardcoded in this notebook, which is committed to a public git
# repo (GitHub's push protection rejects the commit outright, and HF's secret scanner
# auto-revokes any write-scoped token that lands in one).
#
# Easiest on Colab: run this in a scratch cell once per session, paste when prompted:
#     import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

# Resolve the accelerator NOW, before the multi-hour cells below, and print the exact
# torch device that training will use. detect_accelerator() covers both TPU
# generations (legacy XRT env vars and current PJRT ones) plus the /dev/accel* device
# nodes, so a modern Colab/Kaggle TPU runtime is recognised rather than falling through
# to CPU -- a fallback that is otherwise invisible until you notice steps taking 100x
# too long, hours in.
accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


In [ ]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

## 1a. Patch a tokenizer decode bug (inflates val_ar_cer)

The vendored `khmer_segmentation.py` (downloaded above from `Panhapich/khmer-sp-8k`)
has a `KhmerTokenizer.decode()` that does `self.sp.decode(ids).replace(" ", "")` --
this strips **every** space unconditionally, not just the artificial Khmer
word-boundary spaces `segment_line()` introduces for SentencePiece training. So any
real space in the reference text (English words, mixed Khmer/English text,
punctuation spacing) is deleted from the AR decoder's decoded output regardless of
whether the model predicted it correctly -- inflating `val_ar_cer` for any sample
containing genuine whitespace. `val_ctc_cer` is unaffected (`CharVocab.decode` in
`recognizer/data/char_vocab.py` does no space stripping), so it remains a trustworthy
read on encoder accuracy even without this patch.

This cell rewrites the downloaded `khmer_segmentation.py` on disk (the same file
every `KhmerOcrTokenizer()` construction loads from -- see
`khmer_ocr_tokenizer.py`'s `_load_upstream_khmer_tokenizer`) so `decode()` only
strips a space when it falls strictly between two Khmer characters -- the actual
artificial-boundary case -- and leaves every other space alone. Idempotent: skips if
already patched, so re-running this cell (or a fresh fetch that re-downloads the
original file) is safe.

In [ ]:
from recognizer.config import TOKENIZER_ASSETS_DIR

_seg_path = TOKENIZER_ASSETS_DIR / "khmer_segmentation.py"
_seg_src = _seg_path.read_text(encoding="utf-8")

_BUGGY_DECODE = '''    def decode(self, ids) -> str:
        # Strip the artificial word-boundary spaces introduced for training;
        # natural Khmer orthography does not space every word.
        return self.sp.decode(ids).replace(" ", "")'''

_PATCHED_DECODE = '''    def decode(self, ids) -> str:
        # PATCHED (notebook cell 1a): the original body here did
        # `self.sp.decode(ids).replace(" ", "")`, which strips EVERY space --
        # including genuine ones in English words, mixed Khmer/English text, and
        # punctuation spacing -- not just the artificial Khmer word-boundary
        # spaces segment_line() introduces for SentencePiece training. That
        # silently deletes correctly-predicted spaces before CER ever sees them.
        # Only strip a space strictly between two Khmer characters -- the real
        # artificial-boundary case -- and leave every other space alone.
        import re as _re
        decoded = self.sp.decode(ids)
        return _re.sub(r"(?<=[\\u1780-\\u17ff])\\s(?=[\\u1780-\\u17ff])", "", decoded)'''

if _PATCHED_DECODE in _seg_src:
    print(f"{_seg_path} already patched -- nothing to do")
elif _BUGGY_DECODE in _seg_src:
    _seg_path.write_text(_seg_src.replace(_BUGGY_DECODE, _PATCHED_DECODE), encoding="utf-8")
    print(f"patched {_seg_path}: decode() now only strips Khmer-Khmer boundary spaces")
else:
    raise RuntimeError(
        f"{_seg_path} doesn't match the expected buggy decode() body -- the upstream "
        f"file may have changed. Inspect it manually before training: the goal is a "
        f"decode() that doesn't strip every space unconditionally."
    )


## 1b. Low-memory dataset loading patch

`recognizer/data/manifest.py`'s `load_dedup_arrow` (unmodified library code)
materializes the ENTIRE image-bytes column into a Python list
(`table.column("image").to_pylist()`), then builds a second full list of
`Sample` objects from it -- both lists stay alive simultaneously until the
function returns, so peak memory during dataset loading is roughly **2x**
the dataset's actual image-bytes size. On a Colab session this is enough to
get the kernel OOM-killed mid-load, which surfaces as no Python traceback at
all -- just `"Canceled future for execute_request message before replies
were done"` -- because the process itself dies, not one call inside it.

This cell monkeypatches `load_dedup_arrow` to iterate the Arrow columns
directly instead of pre-snapshotting them, so no intermediate full-column
Python list is ever alive alongside the final result -- same output, roughly
half the peak memory. This is a real fix to shared library code, not a
notebook-only workaround -- worth upstreaming into
`recognizer/data/manifest.py` directly once confirmed, so every consumer of
`run_training` benefits, not just this notebook.

In [ ]:
# Monkeypatches recognizer.data.manifest.load_dedup_arrow: safe because
# load_dedup_manifest (which run_training actually calls) looks up
# load_dedup_arrow by name in the module's own namespace at CALL time, not at
# import time -- so reassigning the module attribute here takes effect for
# every call made after this cell runs, without needing to touch train.py or
# re-import anything downstream.
import recognizer.data.manifest as _manifest

def _load_dedup_arrow_low_memory(path):
    import pyarrow as pa

    with pa.memory_map(str(path), "rb") as source:
        table = pa.ipc.open_file(source).read_all()
    text_col = table.column("text")
    source_col = table.column("source")
    image_col = table.column("image")
    # zip() over ChunkedArrays iterates chunk-by-chunk, yielding pa.Scalar
    # objects one at a time -- .as_py() converts just that one value, so at
    # most one row's worth of extra Python objects exists beyond the `samples`
    # list actually being built, vs. the original's three full-column lists
    # PLUS the final list all alive at once.
    samples = []
    for t, s, img in zip(text_col, source_col, image_col):
        samples.append(_manifest.Sample(image_bytes=img.as_py(), text=t.as_py(), source=s.as_py()))
    return samples

_manifest.load_dedup_arrow = _load_dedup_arrow_low_memory
print("patched recognizer.data.manifest.load_dedup_arrow for lower peak memory during dataset loading")

## 2. Data

The full pull -> pack -> dedup pipeline below is expensive (real network transfer +
CPU-bound hashing, potentially a long time at this scale) -- it only needs to run
**once**. The first successful run pushes its result to a private Hugging Face
dataset repo (`real_data.config.HF_DATA_REPO_ID`); every later run (new session, new
notebook, different machine) checks that repo first and just downloads the prebuilt
`dedup.arrow` instead of repeating the pull/pack/dedup work from scratch.

`SAMPLES_PER_SOURCE` only matters the first time (before anything's been pushed to the
Hub). The defaults pull everything available from the three smaller sources, but cap
`chanrith_ocr_image_line` (12M+ rows, ~40GB in full) at 100k rows. Each source is
pulled, packed into a single Arrow file (`<source>.arrow`, image bytes stored exactly
as pulled -- no re-encoding/resizing), and its raw per-image files are deleted before
the next source starts, bounding peak disk usage to "one source's raw files + all
Arrow files packed so far."


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, HF_DATA_REPO_ID, REAL_DATA_ROOT
from real_data import hf_push

# Per-source sample counts for the (one-time) pull from source. Pulls everything
# available from the smaller sources, but caps chanrith_ocr_image_line (12M+ rows) at
# 100k -- pulling it in full would be ~40GB, far more than a Kaggle/Colab session's
# disk budget can hold.
SAMPLES_PER_SOURCE = {
    "deepcopy_khmer_text_recognition": 136_117,
    "chanrith_ocr_image_line": 100_000,
    "darayut_scene_text": 102_500,
    "sokheng_synthetic_v1": 100_000,
}

# KMP_DUPLICATE_LIB_OK/OMP_NUM_THREADS: Colab/Kaggle commonly have more than one
# OpenMP runtime on the import path (numpy, PIL/imagehash, datasets' native deps each
# bundle their own libomp/libiomp5) -- loading two in one process is a well-known cause
# of an immediate SIGABRT with zero output, right at import time, before any of this
# script's own code runs. Setting these before spawning avoids that class of crash;
# harmless if it wasn't actually the cause.
SUBPROCESS_ENV = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1"}

def run_checked(cmd):
    """Runs `cmd`, always printing its output, and raises with the actual captured
    stderr on failure -- a bare `subprocess.CalledProcessError` (or, worse, a `!shell`
    cell whose exit code isn't checked at all) hides exactly the text that explains
    *why* it died, which is the difference between a one-line fix and a guessing game."""
    result = subprocess.run(cmd, env=SUBPROCESS_ENV, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        sys.stderr.write(result.stderr)
        raise RuntimeError(
            f"command failed (exit code {result.returncode}"
            f"{', likely killed by a signal -- see stderr above for the real cause' if result.returncode < 0 else ''}"
            f"): {' '.join(cmd)}"
        )
    return result

dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup.arrow"
built_locally = False  # tracks whether THIS run built dedup_manifest from source
                        # (vs. it already being local, or downloaded from the Hub) --
                        # only push to the Hub in the first case (section 2b below).

if dedup_manifest.exists():
    print(f"{dedup_manifest} already present locally, skipping pull/download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"found a prebuilt dataset on the Hub ({HF_DATA_REPO_ID}) -- downloading "
          f"instead of re-pulling/re-deduplicating from source")
    hf_push.pull_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID)
else:
    print(f"no prebuilt dataset found on {HF_DATA_REPO_ID} -- pulling + packing from "
          f"source (one-time cost; result gets pushed to the Hub in the next cell)")
    built_locally = True

    # Pull -> pack to Arrow -> delete raw, one source at a time (not all sources
    # pulled first, then packed): this bounds peak disk usage to "current source's
    # raw files + every Arrow file packed so far", instead of needing all 4 sources'
    # raw files on disk simultaneously.
    arrow_files = []
    for source in EXTERNAL_DATASETS:
        arrow_path = REAL_DATA_ROOT / "samples" / f"{source}.arrow"
        arrow_files.append(arrow_path)
        if arrow_path.exists():
            print(f"{source}: already packed, skipping")
            continue

        source_dir = REAL_DATA_ROOT / "samples" / source
        num_samples = SAMPLES_PER_SOURCE[source]
        if not (source_dir / "manifest.tsv").exists():
            print(f"{source}: pulling {num_samples} samples...")
            run_checked([sys.executable, "-m", "real_data.generate_external_chunks",
                         "--source", source, "--num-samples", str(num_samples)])

        print(f"{source}: packing to {arrow_path}...")
        run_checked([sys.executable, "-m", "real_data.pack_arrow",
                     "--source", source, "--delete-raw"])

    print(arrow_files)


In [ ]:
# 2b. Deduplicate + push to the Hub -- only runs if this session actually built the
# dataset from source above (built_locally == True); a no-op if dedup_manifest was
# already local or was just downloaded from the Hub.
if built_locally:
    # --near-dup-threshold 0 disables the O(n^2) near-dup pass -- REQUIRED at this
    # scale (hundreds of thousands of rows): the default pairwise comparison is
    # O(n^2) and would take an impractically long time (the nonzero default is only
    # tuned/safe for the notebook-scale hundreds-to-thousands range, e.g.
    # notebooks/train_diagnose_eval.ipynb).
    missing = [str(p) for p in arrow_files if not p.exists()]
    if missing:
        raise RuntimeError(
            "The following sources are missing their packed .arrow file -- re-run "
            "the pull cell above (in full, for all 4 sources) before deduplicating. "
            "This usually means the runtime restarted/reset between the pull and "
            "dedup cells (e.g. after a crash) and the previously-pulled data under "
            f"{REAL_DATA_ROOT} was lost:\n  " + "\n  ".join(missing)
        )

    run_checked([sys.executable, "-m", "real_data.deduplicate",
                 "--arrow-files", *[str(p) for p in arrow_files],
                 "--out", str(dedup_manifest),
                 "--near-dup-threshold", "0"])
    assert dedup_manifest.exists(), (
        f"{dedup_manifest} was not created -- check the pull cell above actually "
        f"populated {[str(p) for p in arrow_files]} before dedup ran."
    )
    print("dedup arrow file ready:", dedup_manifest)

    print(f"pushing prebuilt dataset to the Hub ({HF_DATA_REPO_ID}) so future runs "
          f"can skip straight to downloading it...")
    url = hf_push.push_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID, private=True)
    print("pushed:", url)
else:
    print(f"{dedup_manifest} already ready (local or from the Hub) -- nothing to dedup/push")


## 2c. Data quality filter

One of the three fixed diagnostic samples tracked during a prior training run
(ground truth `'វិរាគចិត្ត ២០០២៛\nជោះ 8x10,000'`, with a literal newline in the
transcript) produced garbage predictions across 8,000+ straight logged steps --
a strong sign the image is a single cropped line but the transcript spans two,
not something any amount of training fixes. `find_unlearnable` (run inside
`run_training`, next section) only drops samples whose CTC target is longer
than the encoder frames the image can produce -- it does not catch this
different failure mode, where the transcript simply does not correspond to
what's pictured.

This cell scans `dedup.arrow` for embedded newlines (the cheapest,
highest-confidence signal available without per-sample manual review) and
writes a filtered copy, `dedup_filtered.arrow`, that the training cell uses
instead. Runs once and is cached like every other data-prep step in this
notebook -- if you add other mismatch heuristics later, delete
`dedup_filtered.arrow` to force a rebuild.

In [ ]:
import pyarrow as pa

dedup_filtered = dedup_manifest.parent / "dedup_filtered.arrow"

# Validity, not just existence: a prior interrupted write (Colab disconnect,
# kernel restart, out-of-memory mid-write) can leave a truncated/corrupt file
# behind, and pa.ipc.open_file on a corrupt file raises "ArrowInvalid: Not an
# Arrow file" much later, inside run_training -- confusing, since by then it
# looks like a training-cell bug rather than a leftover bad file from this
# cell. Checking exists() alone (as an earlier version of this cell did) trusts
# that leftover file forever, since it never gets rewritten once present.
def _is_valid_arrow_file(path):
    if not path.exists():
        return False
    try:
        with pa.memory_map(str(path), "rb") as f:
            pa.ipc.open_file(f).schema
        return True
    except pa.ArrowInvalid:
        return False

if _is_valid_arrow_file(dedup_filtered):
    print(f"{dedup_filtered} already present and valid, skipping filter pass")
else:
    if dedup_filtered.exists():
        print(f"{dedup_filtered} exists but is not a valid Arrow file "
              f"(likely an interrupted write from a previous session) -- rebuilding")
    print(f"scanning {dedup_manifest} for transcript/image line-count mismatches...")
    with pa.memory_map(str(dedup_manifest), "rb") as source:
        table = pa.ipc.open_file(source).read_all()

    texts = table.column("text").to_pylist()
    sources = table.column("source").to_pylist()

    # A literal "\n" in a transcript is the cheapest, highest-confidence signal
    # that the label spans more lines than the (single-line-cropped) image
    # actually shows -- no gradient update can fix a target that doesn't match
    # its image. Extend this predicate if other mismatch patterns turn up.
    keep_mask = [("\n" not in t) for t in texts]
    dropped_by_source = {}
    for t, s, keep in zip(texts, sources, keep_mask):
        if not keep:
            dropped_by_source[s] = dropped_by_source.get(s, 0) + 1

    n_total = len(texts)
    n_dropped = n_total - sum(keep_mask)
    print(f"dropping {n_dropped}/{n_total} samples with embedded newlines "
          f"(likely multi-line transcript vs single-line image): {dropped_by_source or 'none'}")

    filtered_table = table.filter(pa.array(keep_mask))
    # Write to a .tmp path and atomically rename into place on success -- same
    # pattern dataset.py's compute_widths uses for its widths_cache.json, so an
    # interrupted write (Colab disconnect, OOM, kernel restart) never leaves a
    # half-written dedup_filtered.arrow sitting at the real filename for a later
    # run to mistake for a finished, valid file.
    tmp_path = dedup_filtered.with_name(dedup_filtered.name + ".tmp")
    with pa.OSFile(str(tmp_path), "wb") as sink:
        with pa.ipc.new_file(sink, filtered_table.schema) as writer:
            writer.write_table(filtered_table)
    tmp_path.replace(dedup_filtered)
    print(f"wrote filtered dataset -> {dedup_filtered}")

dedup_manifest = dedup_filtered
print("training will use:", dedup_manifest)

## 3. Train

In [ ]:
from pathlib import Path

from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training
from recognizer.hf_push import pull_latest_checkpoint

# max_tokens_per_block=16, same as v2_scratch_k16's current config -- irrelevant to
# this run's actual training (blockwise mode is never reached, see sequential_ar_steps
# below) but kept identical so the model architecture matches v2_scratch_k16 exactly.
model_cfg = ModelConfig(max_tokens_per_block=16)

LOG_EVERY = 100
NUM_WORKERS = 0
CKPT_EVERY = 2_000
MAX_STEPS = 80_000

train_cfg = TrainConfig(
    log_every=LOG_EVERY,
    num_workers=NUM_WORKERS,
    ckpt_every=CKPT_EVERY,
    max_eval_samples=512,      # val samples the periodic eval covers for CTC CER.
    max_ar_eval_samples=64,    # AR greedy-decode eval sample cap -- sequential AR
                                # decode is one token per forward pass, not cheap.
    ctc_weight=0.5,            # same fixed value every v2-family notebook uses.
    # Equal to MAX_STEPS: step < sequential_ar_steps holds for the WHOLE run, so the
    # AR decoder never leaves teacher-forced sequential mode (see decoder.py's
    # forward_sequential) -- this run never enters blockwise mode at all, unlike
    # train_recognizer_v2_scratch.ipynb's sequential_ar_steps=60_000 (a phase, not
    # the whole run).
    sequential_ar_steps=MAX_STEPS,
    max_steps=MAX_STEPS,
)

RUN_NAME = "v2_ctc_ar_sequential"
CHECKPOINT_REPO_ID = "Panhapich/Tuna-OCR"
HUB_PATH_PREFIX = "ctc_ar_sequential"

resume_path = pull_latest_checkpoint(Path(checkpoint_root) / RUN_NAME, token=hf_token,
                                     repo_id=CHECKPOINT_REPO_ID, path_prefix=HUB_PATH_PREFIX)
if resume_path:
    print(f"resuming from the latest checkpoint on the Hub: {resume_path}")
else:
    print(f"no Hub checkpoint found for {CHECKPOINT_REPO_ID}/{HUB_PATH_PREFIX} -- starting {RUN_NAME} from scratch")

model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,  # points at dedup_filtered.arrow -- see section 2c.
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=True,
    repo_id=CHECKPOINT_REPO_ID,
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,  # OOM-probing auto-tune. CUDA only -- on TPU/CPU the
                       # configured TrainConfig.batch_size is used as-is, since XLA
                       # compiles lazily and never raises a catchable Python OOM.
    resume_path=resume_path,
    hub_path_prefix=HUB_PATH_PREFIX,
)